In [ ]:
import pandas as pd
import numpy as np
from math import pi

In [ ]:
import sys
print(sys.executable)

In [ ]:
food = pd.read_csv("Emergency_Food_and_Meals_Seattle_and_King_County_20260222.csv")

food.head()

,Food Resource Type,Agency,Location,Operational Status,Who They Serve,Address,Latitude,Longitude,Phone Number,Website,Days/Hours,Date Updated
0,Food Bank,Algona/Pacific Food Pantry,Algona/Pacific Food Pantry - Food Distribution...,Open,General Public,"603 3rd Ave SE, Pacific, WA 98047",47.265011,-122.236771,(253)-351-0450,https://networks.whyhunger.org/organisation/41599,Tuesday,7/22/2025
1,Food Bank & Meal,Asian Counseling and Referral Service (ACRS),ACRS Food Bank and Meals,Open,General Public,"800 S Weller St, Seattle, WA 98004",47.584363,-122.330378,(206)-695-7510,https://acrs.org/services/aging-services-for-o...,"Wednesday, Friday",8/7/2025
2,Meal,Asian Counseling and Referral Service (ACRS),Club Bamboo and Hmong Senior Club,Open,Older Adults 60+ and Eligible Participants,"3639 Martin Luther King Jr Way S, Seattle, WA ...",47.570897,-122.297388,(206)-695-7510,https://www.acrs.org/services/aging-services-f...,"Monday, Tuesday, Wednesday, Thursday, Friday",8/7/2025
3,Meal,Asian Counseling and Referral Service (ACRS),Duoc Su Senior Nutrition Program,Open,Older Adults 60+ and Eligible Participants,"6924 42nd Ave S , Seattle, WA 98118",47.540000,-122.280317,(206)-695-7510,https://www.acrs.org/services/aging-services-f...,Saturday,7/21/2025
4,Meal,Asian Counseling and Referral Service (ACRS),Friendly Island of Tonga Seniors,Open,Older Adults 60+ and Eligible Participants,"4620 SW Graham St, Seattle, WA 98136",47.546979,-122.392267,(206)-695-7510,https://www.acrs.org/services/aging-services-f...,NaN,7/22/2025


In [ ]:
food.info()

In [5]:
food.describe()
food.isnull().sum()

Food Resource Type     0
Agency                 0
Location               0
Operational Status     0
Who They Serve         0
Address                0
Latitude              36
Longitude             36
Phone Number          22
Website                6
Days/Hours            27
Date Updated           0
dtype: int64

In [6]:
!pip install geopy

In [7]:
from geopy.geocoders import Nominatim
from time import sleep

In [8]:
geolocator = Nominatim(user_agent="food_access_project")

for idx, row in food[food["Latitude"].isnull()].iterrows():
    location = geolocator.geocode(row["Address"])
    if location:
        food.at[idx, "Latitude"] = location.latitude
        food.at[idx, "Longitude"] = location.longitude
    sleep(1)

In [9]:
food.isnull().sum()

Food Resource Type     0
Agency                 0
Location               0
Operational Status     0
Who They Serve         0
Address                0
Latitude               2
Longitude              2
Phone Number          22
Website                6
Days/Hours            27
Date Updated           0
dtype: int64

In [10]:
food = food.dropna(subset=["Latitude", "Longitude"])

In [11]:
food["Phone Number"] = food["Phone Number"].fillna("Not Available")
food["Website"] = food["Website"].fillna("Not Available")
food["Days/Hours"] = food["Days/Hours"].fillna("Unknown")

In [12]:
food["Food Resource Type"] = food["Food Resource Type"].str.lower().str.strip()
food["Operational Status"] = food["Operational Status"].str.lower()

In [13]:
food["Address"] = food["Address"].str.title().str.strip()

In [14]:
food1 = food[
    (food["Latitude"] > 47) & 
    (food["Latitude"] < 48)
]

In [15]:
food1 = food1.drop_duplicates()

In [16]:
food1['bank_id'] = 'foodbank' + food1.index.astype(str)

In [17]:
def lat_lon_to_mercator(lat, lon):
    """Convert lat/lon to Web Mercator projection"""
    k = 6378137
    x = lon * (k * pi/180.0)
    y = np.log(np.tan((90 + lat) * pi/360.0)) * k
    return x, y

In [18]:
food1['x'], food1['y'] = lat_lon_to_mercator(
        food1["Latitude"].values,
        food1["Longitude"].values
    )

In [19]:
food1.to_csv("./clean_food_resources.csv", index=False)

In [20]:
king_county = pd.read_csv("/Users/sowmyadadheech/Documents/DATA 515/project/data/clean_king_county_stops.csv")
king_county.head()


FileNotFoundError: [Errno 2] No such file or directory: '/Users/sowmyadadheech/Documents/DATA 515/project/data/clean_king_county_stops.csv'

In [3]:
all_shapes = pd.read_csv('/Users/sowmyadadheech/Documents/DATA 515/project/data_preprocessing/processed/all_shapes.csv')
all_shapes

/var/folders/w1/p0zs1nx950d163p_vsm_rkk80000gn/T/ipykernel_15534/2048406447.py:1: DtypeWarning: Columns (0: shape_id) have mixed types. Specify dtype option on import or set low_memory=False.
  all_shapes = pd.read_csv('/Users/sowmyadadheech/Documents/DATA 515/project/data_preprocessing/processed/all_shapes.csv')


,shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled
0,10002005,47.612137,-122.281769,1,0.0
1,10002005,47.612144,-122.281784,2,5.8
2,10002005,47.612148,-122.281830,3,13.5
3,10002005,47.612141,-122.281853,4,22.0
4,10002005,47.612102,-122.281921,5,45.0
...,...,...,...,...,...
180254,T25:T01,47.239589,-122.429978,8132,20759.9
180255,T25:T01,47.239629,-122.429631,8133,20847.3
180256,T25:T01,47.239698,-122.429249,8134,20945.5
180257,T25:T01,47.239791,-122.428502,8135,21134.1
